In [3]:
from keithley_apis import *
import zhinst

import numpy as np
import matplotlib.pyplot as plt
import time
import zhinst.core
from datetime import datetime
import pyvisa as visa
from datetime import datetime
import json
import os

from RFSoC_PARAMPconfig_08012024_dysl_v1 import *

In [4]:
ip = "TCPIP0::192.168.0.27::inst0::INSTR"
rm = visa.ResourceManager()
kna = rm.open_resource(ip)

In [5]:
# daq = zhinst.core.ziDAQServer('127.0.0.1', 8004, 6)
keithley_ip = "TCPIP0::192.168.0.125::inst0::INSTR"
daq = rm.open_resource(keithley_ip)

In [6]:
#%% Configure the board

# # Create board object 
board = RFSoC_auxill("192.168.0.210",userconfig=None,debug=True,logging=False)

Connection Established.
Version 2020.2
RfdcVersion rfdc_v8.1
RFSoC PARAMP CLASS


228 is DAC tile 0

229 is DAC tile 1

230 is DAC tile 2

231 is DAC tile 3


Each Tile has 4 DACs - 0,1,2,3

Name convention on board

0_228 => DAC 0 of Tile 0

2_229 => DAC 2 of Tile 1

1st index on all list of lists indicate tile

2nd index on all list of lists indicate DAC



In [7]:
# DAC Tile and HDAWG DAC defs

#[[Tile,DAC],HDAWG_DAC]

ch_map = { "rr1": [[0,0],1],
          "rr2": [[1,2],2],
          "rr3": [[1,0],3],
          "rr4": [[0,2],4],
          "rr5": [[0,1],5],
          "rr6": [[0,3],6],
          "rr7": [[1,3],7],
          "rr8": [[1,1],8]
}

In [ ]:
board.establish_connection()



In [ ]:
# daq.close()
board.break_connection()

NameError: name 'board' is not defined

: 

# just run the following w/o thinking

In [8]:
#%% USER AND BOARD PARAMETERS

# DUAL TONE PROPERTY DECLARATION
# IF PLAN
# f_c = [[0e6,0e6,0e6,0e6],[0e6,0e6,0e6,0e6]] # Centre Frequency declaration. 
f_sep = [[1600e6,1600e6,1600e6,1600e6],[1600e6,1600e6,1600e6,1600e6]] # Frequency Separation of dual tones

# DAC TILE and BLOCK PLAN
Tiles = [[0, 0, 0, 0],[1, 1, 1, 1]]
Blocks = [[0, 1, 2, 3],[0, 1, 2, 3]]
Type = 1

# BOARD DAC FIFO SAMPLING RATES
F_sample = 8847.36#8355.84 # MHz # RF sampling Frequency

F_sample1 = [[8847.36,8847.36,8847.36,8847.36]]

Interpolation = 2 # 4: 2x inerpolation
fs_IF = F_sample*1e6/Interpolation # Sampling Frequency of IF
# BOARD DAC PROPERTIES (FOR DIRECT SETTING)
# f_firstNyq = 1355.84 # MHz

Cavity_freqs = [[0,0,0,0],[0,0,0,0]]


# PYTHON CODE PARAMETERS
wait_in_sec = 0.0
# NUMBER OF SAMPLES IN I OR Q CHANNEL OF BRAM
N = 8192 * 1
# PREALLOCATION
wave_data = [[0,0,0,0],[0,0,0,0]]        

In [57]:
#Cavity Frequency Setting

# in MHz

cavity_frequencies = {
    "rr1": 7.394663e3,
    "rr2": 7.42342e3,
    "rr3": 7.493969e3,
    "rr4": 7.395136e3,
    "rr5": 7.405661e3,
    "rr6": 7.344403e3,
    "rr7": 7.12446e3,
    "rr8": 7.41618e3,
}

In [10]:
cavity_freq_extract(cavity_frequencies=cavity_frequencies)

[[7394.663, 7423.42, 7493.969, 7395.136],
 [7405.661, 7344.403, 7124.46, 7416.18]]

Cavity Frequency Setting

In [11]:
# AMPLITUDE PLAN
Amp1_offset = [[0,0,0,0],[0,0,0,0]] # DC Offset
Amp1 = [[0.5,0.5,0.43,0.43],[0.38,0.45,0.4,0.55]]        # Sideband imbalance
tot_pwr = [[0.0,0.0,0.275,0.275],[0,0.0,0.0,0.0]]

# DAC CURRENT PLAN
DAC_currents = [[14.75,14.5,21.25,31.75],[28.75,20,14,20]]       # currents in mA

DC_bias = [[-1.1,0,0,0],[0,0,0,0]]



# cavity_freqs = [[7.41618e3,7.3e3,4.5e3,7.3e3],[7.41618e3,7.3e3,7.3e3,7.3e3]]
# cavity_freqs = cavity_freq_extract(cavity_frequencies=cavity_frequencies)

# for i in range(2):
#     for j in range(4):
#         Cavity_freqs[i][j] = F_sample - cavity_freqs[i][j]

for rr, freq in cavity_frequencies.items():
    t_dac = ch_map[f'rr{rr[-1]}'][0][1]
    t_tile = ch_map[f'rr{rr[-1]}'][0][0]
    # print(f't_dac = {t_dac} t_tile = {t_tile}')
    Cavity_freqs[t_tile][t_dac] = F_sample - freq

## Load settings from file

In [ ]:
date = '25-02-27_'

qubits = [i[-1] for i in cavity_frequencies.keys()]

data_file_path = './paramp_data_files/paramp_gain_data_'

for q in qubits:

    q = int(q)
    fin_path = data_file_path + date + f'R{(2*q+5)%12}.json'

    flg = os.path.isfile(fin_path)

    if flg:
        with open(fin_path, 'r') as f:
            data_json = json.load(f)
            f.close()

        DAC_no = ch_map[f'rr{q}'][0][1]
        Tile = ch_map[f'rr{q}'][0][0]


        hdawg_daq_ch = ch_map[f'rr{q}'][1]
        
        DAC_currents[Tile][DAC_no] = data_json['DAC_currents']
        Amp1_offset[Tile][DAC_no] = 0
        tot_pwr[Tile][DAC_no] = data_json['tot_pwr']
        Amp1[Tile][DAC_no] = data_json['Amp1']
        DC_bias[Tile][DAC_no] = data_json['DC_bias']

In [ ]:
DC_bias

In [ ]:
DAC_currents

# int for Internal source and ext for External Source

In [ ]:
board_init(board=board,int_ext='ext',F_sample=F_sample) # Only touch the int ext for clock source

# just run this w/o thinking

In [12]:
#%% GENERATE DATA (just run this w/o thinking)

for i in range(len(Blocks)):
    for j in range(len(Blocks[i])):
        [wave_data[i][j],_,_] = gen_pulses(fs_IF, N, 0, f_sep[i][j], Amp1_offset[i][j], Amp1[i][j],tot_pwr[i][j])

# Programming beings. Set rr number and pump = 1/0 for on/off

* # Initialisation. Start with tot_pwr = 0.5 and Amp1 = 0.5. Amp1 is the lower sideband power fraction i.e. if Amp1=1, then signal-wise same power in both sidebands. If DAC current adjustment reaches 32, and still not enough gain, increase tot_pwr and start lower again.

* # f_sep = 0 for single pump and f_sep = 1600e6 for double pump at 800e6 gap

* # Can optimise sideband power ratio to get same gain at lower pump power

* # Saturates at tot_pwr1 = 0.28 and DAC Current = 32 mA for all DACs. TAKE CARE 

In [13]:
#%% PROGRAM DAC one by one

rr = 2

#[[Tile,DAC],HDAWG_DAC]

DAC_no = ch_map[f'rr{rr}'][0][1]
Tile = ch_map[f'rr{rr}'][0][0]


hdawg_daq_ch = ch_map[f'rr{rr}'][1]

# current = 10


In [ ]:
DAC_currents[Tile][DAC_no]

In [ ]:
#### Saturates at tot_pwr1 = 0.25 for all DACs. TAKE CARE
pump = 0#pump on/off
print(f'Pump state = {pump}')

if pump:
    tot_pwr_adj = 0.2
    Amp1_adj = 0.48
    [data,_,_] = gen_pulses(fs_IF=fs_IF,N=N, f_c=0, f_sep=1600e6, Amp1_offset=0.0, Amp1=Amp1_adj,tot_pwr1=tot_pwr_adj)##DOUBLE PUMP
    # [data,_,_] = gen_pulses(fs_IF=fs_IF,N=N, f_c=0, f_sep=0, Amp1_offset=0.0, Amp1=0.48,tot_pwr1=0.275)##SINGLE PUMP
    # DAC_prog(board, 1, Tile, DAC_no, F_sample, 32, Cavity_freqs[DAC_no][Tile], data)
    DAC_prog(board, 1, Tile, DAC_no, F_sample, current, Cavity_freqs[Tile][DAC_no], data)
else:
    [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, 0.5,0.0)
    DAC_prog(board, 1, Tile, DAC_no, F_sample, 8, Cavity_freqs[Tile][DAC_no], data)
    

Pump state = 1
 Programming DAC 2_230 (DAC 2, Tile 1)...
SetMemType
SetDataPathMode 1 2 2 
SetMMCM 1 5 2 5 0 
SetNyquistZone
SetupFIFO
SetInterpolationFactor
SetMMCM 1 5 2 5 0 
IntrClr
SetupFIFO
MultiBand 1 1 1 4 1
SetMixerSettings
ResetNCOPhase
UpdateEvent
SetDACVOP
SetDecoderMode
SetInvSincFir
DAC 2_230 (DAC 2, Tile 1) settings programmed.
Sending data to DAC 2, Tile 1. (2_230)
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
LocalMemTrigger
SetLocalMemSample
------------------------------COMMAND SENT----------------------------
-----------------------------DATA SENT--------------------------------
WriteDataToMemory
LocalMemTrigger
LocalMemTrigger
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
Data Sent. 2_230 DAC active (DAC 2, Tile 1)
SetDACVOP
DATA UPLOADED


# HDAWG offset adjustment

# Current Adjustment

In [143]:
#%% POST CALIBRATION STATUS CHECK / UPDATE
current = 19.75
change_current(board,current,DAC_no,Tile)
print(f'Current current = {current}')

SetDACVOP
Current current = 19.75


In [ ]:
daq.close()

In [144]:
offset = 0
# (daq, ch, V_fin, V_init=None, step=0.01, t_wait=1, verbose=False):
ramp_V_hdawg(daq,ch=hdawg_daq_ch, step = 0.2, V_fin = offset)

V_init automatically taken from current setting of 14.4
values 14.4, 0, 0.2, 14.4
14.4
14.2
14.0
13.8
13.6
13.4
13.2
13.0
12.8
12.6
12.4
12.2
12.0
11.8
11.6
11.4
11.2
11.0
10.8
10.6
10.4
10.2
10.0
9.8
9.6
9.4
9.2
9.0
8.8
8.6
8.4
8.2
8.0
7.8
7.6
7.4
7.2
7.0
6.8
6.6
6.4
6.2
6.0
5.8
5.6
5.4
5.2
5.0
4.8
4.6
4.4
4.2
4.0
3.8
3.6
3.4
3.2
3.0
2.8
2.6
2.4
2.2
2.0
1.8
1.6
1.4
1.2
1.0
0.8
0.6
0.4
0.2
0.0
Current


In [ ]:
daq.write("SOURce:VOLT:ILIM 0.05")

In [ ]:
# Save settings


DAC_currents[Tile][DAC_no] = current
tot_pwr[Tile][DAC_no] = tot_pwr_adj
Amp1[Tile][DAC_no] = Amp1_adj
Cavity_freqs[Tile][DAC_no]

# DC_bias[Tile][DAC_no] = read_offset(daq,hdawg_daq_ch) + read_amp(daq,hdawg_daq_ch)

In [ ]:
DAC_currents, DC_bias, Cavity_freqs[Tile][DAC_no], F_sample -Cavity_freqs[Tile][DAC_no], fs_IF

In [ ]:
## Save with VNA data into file

gain_profile = (kna.query_ascii_values("CALC1:MEAS1:DATA:FDAT?"))
time.sleep(4)
f_data = (kna.query_ascii_values("CALC1:MEAS1:X:VAL?"))

gain_data = list(np.transpose([f_data, gain_profile]))

save_data = {}
save_data['DAC_currents'] = current
save_data['tot_pwr'] = tot_pwr_adj
save_data['Amp1'] = Amp1_adj
save_data['DC_bias'] = offset
save_data['Cavity_freq'] = Cavity_freqs[Tile][DAC_no]


now = datetime.now()
current_date = now.strftime("%y-%m-%d")

file_name = f'paramp_gain_data_{current_date}_R{(2*rr+5)%12}'

with open(f'./paramp_data_files/{file_name}.json','w') as f:
    json.dump(save_data, f, indent = 6)
    f.close()

np.savetxt(fname = f'./paramp_data_files/{file_name}.csv', X = gain_data)

In [ ]:
rr

In [ ]:
save_data

In [ ]:
#Print final settings
DAC_currents, tot_pwr, Amp1, DC_bias


In [ ]:
## All pymp off

for i in range(2):
    for j in range(4):
        [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, 0.5,0.0)
        DAC_prog(board, 1, Tile, DAC_no, F_sample, 8, Cavity_freqs[i][j], data)

In [145]:
board.break_connection()

Connection Closed.


In [146]:
daq.close()

In [ ]:
board.establish_connection()
daq.open()

In [ ]:
# AMPLITUDE PLAN
Amp1_offset = [[0,0,0,0],[0,0,0,0]] # DC Offset
Amp1 = [[0.5,0.5,0.43,0.43],[0.38,0.45,0.4,0.55]]        # Sideband imbalance
tot_pwr = [[0.0,0.0,0.275,0.275],[0,0.0,0.0,0.0]]

# DAC CURRENT PLAN
DAC_currents = [[14.75,14.5,21.25,31.75],[28.75,20,14,20]]       # currents in mA

In [ ]:
## All pymp on simultaneously

for i in range(2):
    for j in range(4):
        [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, Amp1[i][j],tot_pwr[i][j])
        DAC_prog(board, 1, Tile, DAC_no, F_sample, DAC_currents[i][j], Cavity_freqs[i][j], data)

all_ch_off(daq)